# Replicate the CMLP Baseline

This notebook prepares a small, editable workflow for building and testing a quaternion CMLP yourself. It uses the project implementation first, then gives you a compact notebook-only version that mirrors the same equations.

## Setup

The cells below make the local `src/` package importable whether the notebook is launched from the repo root or from `notebooks/`.

In [ ]:
from pathlib import Path
import os
import sys
import time
import matplotlib.pyplot as plt
import torch
import itertools


In [ ]:

mpl_config = Path("/tmp/matplotlib")
mpl_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config))

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
repo_root = next(path for path in candidates if (path / "src" / "qvnn_pc").exists())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
from qvnn_pc.data import make_benchmark_regression_split
from qvnn_pc.functions import ackley_4d, griewank_4d, rastrigin_4d, schwefel_4d
from qvnn_pc.models import CMLPQuaternionMLP, BENCHMARK_CMLP_HYPERPARAMETERS, scalar_regression_metrics
from qvnn_pc.activations import split_sigmoid, split_sigmoid_derivative, split_swish, split_swish_derivative
from qvnn_pc.quaternion import q_conj, q_hadamard, q_matmul, q_mul, q_norm_sq


In [ ]:

torch.set_default_dtype(torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Benchmark Data

A 4D real input is encoded as one quaternion: `[x1, x2, x3, x4]`. Targets are encoded as `f(x) + 0i + 0j + 0k`, so evaluation uses only the real component.

In [ ]:
FUNCTIONS = {
    "ackley": ackley_4d,
    "griewank": griewank_4d,
    "rastrigin": rastrigin_4d,
    "schwefel": schwefel_4d,
}

BOUNDS = {
    "ackley": (-30.0, 30.0),
    "griewank": (-600.0, 600.0),
    "rastrigin": (-5.12, 5.12),
    "schwefel": (-500.0, 500.0),
}


In [ ]:
function_name = "griewank"
n = 30000
sample_seed = 1
split_seed = 2

split = make_benchmark_regression_split(
    function_name,
    n=n,
    sample_seed=sample_seed,
    split_seed=split_seed,
    device=device,
)

x_train, y_train = split.x_train, split.y_train
x_val, y_val = split.x_val, split.y_val
x_test, y_test = split.x_test, split.y_test

x_train.shape, y_train.shape, split.x_mean, split.x_std, BENCHMARK_CMLP_HYPERPARAMETERS[function_name]

In [ ]:
# Viz the Function as a 3D surface plot (fixing x3 and x4 to 0)
n_points_per_dim = 100
x1 = torch.linspace(BOUNDS[function_name][0], BOUNDS[function_name][1], n_points_per_dim)
x2 = torch.linspace(BOUNDS[function_name][0], BOUNDS[function_name][1], n_points_per_dim)
x3 = torch.zeros(n_points_per_dim)  # Fix x3 to 0
x4 = torch.zeros(n_points_per_dim)  # Fix x4 to 0

# 3D Plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
X1, X2 = torch.meshgrid(x1, x2, indexing="ij")
X_grid = torch.stack([X1, X2, x3.unsqueeze(0).expand_as(X1), x4.unsqueeze(0).expand_as(X1)], dim=-1)
y_grid = FUNCTIONS[function_name](X_grid)
ax.plot_surface(X1.cpu(), X2.cpu(), y_grid.cpu(), cmap="viridis")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("f(x1, x2, 0, 0)")
ax.set_title(f"{function_name} function (x3=0, x4=0)")
plt.show()

## Use the Project CMLP

This is the baseline implementation from `src/qvnn_pc/models/cmlp.py`: architecture `[1, 5, 5, 5, 1]`, manual CMLP deltas, and quaternion Adadelta accumulators.

In [ ]:
model = CMLPQuaternionMLP.for_benchmark(
    function_name,
    generator=torch.Generator(device=device.type).manual_seed(42),
    device=device,
)

print(model)
print("layer sizes:", model.layer_sizes)
print("weight shapes:", [tuple(w.shape) for w in model.weights])
print("bias shapes:", [tuple(b.shape) for b in model.biases])

In [ ]:
print("standardized train preactivations")
standardized_diag = model.print_preactivation_diagnostics(x_train[:256])

print("\nraw train preactivations")
raw_x_train = split.raw_x_train[:256].unsqueeze(1)
raw_diag = model.print_preactivation_diagnostics(raw_x_train)


In [ ]:
with torch.no_grad():
    activations, preacts = model.forward_with_cache(x_train[:8])
    y_pred = activations[-1]

print("prediction shape:", tuple(y_pred.shape))
print("activation shapes:", [tuple(x.shape) for x in activations])
print("pre-activation shapes:", [tuple(s.shape) for s in preacts])

deltas = model.compute_deltas(activations, preacts, y_train[:8])
dW, db = model.compute_update_directions(activations, deltas)
print("delta shapes:", [tuple(delta.shape) for delta in deltas])
print("dW shapes:", [tuple(direction.shape) for direction in dW])
print("db shapes:", [tuple(direction.shape) for direction in db])

## Train and Evaluate

Start with a small number of epochs so iteration stays quick. Increase `epochs` once the shapes and metrics look sensible.

In [ ]:
epochs = 250
history = model.fit_online_source_style(
    x_train, 
    y_train, 
    x_val=x_val, 
    y_val=y_val, 
    epochs=epochs
)

In [ ]:
test_metrics = model.evaluate(x_test, y_test)

print(f"runtime: {history['runtime']:.3f}s")
print("final train mse:", history["mse"][-1])
print("final validation loss:", history["validation_loss"][-1])
for metric_name, metric_value in test_metrics.items():
    print(f"test {metric_name}: {metric_value}")

In [ ]:
import matplotlib.pyplot as plt

# Erstellt ein Layout: 2 Zeilen, 3 Spalten
# Spalte 0 ist für den Loss, Spalten 1 & 2 für das 2x2 Grid
plt.figure(figsize=(16, 8))

# 1. Der MSE Plot (nimmt die gesamte linke Spalte ein)
plt.plot(history["mse"], label="train MSE")
plt.plot(history["validation_loss"], label="validation MSE")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.yscale("log")
plt.title("Training Progress")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Plot component-wise predictions vs true values (2x2 Grid)
fig = plt.figure(figsize=(12, 10))
with torch.no_grad():
    # Annahme: model(x_test) gibt (N, 4) zurück
    all_preds = model(x_test).cpu() 
    all_true = y_test.cpu()

for i in range(4):
    # Berechnet Position im 2x2 Grid (Spalten-Offset +1)
    row = i // 2
    col = (i % 2) + 1
    ax = fig.add_subplot(2, 2, i + 1)
    
    pred = all_preds[..., i].flatten()
    true = all_true[..., i].flatten()
    
    ax.scatter(true, pred, s=8, alpha=0.5)
    
    # Diagonale Linie
    limits = [min(true.min(), pred.min()).item(), max(true.max(), pred.max()).item()]
    ax.plot(limits, limits, color="black", linewidth=1, linestyle="--")
    
    ax.set_title(f"Component x{i+1}")
    ax.set_xlabel("True")
    ax.set_ylabel("Pred")

plt.tight_layout()
plt.show()

In [ ]:
n_points_per_dim = 50
low, high = BOUNDS[function_name]
x1 = torch.linspace(low, high, n_points_per_dim)
x2 = torch.linspace(low, high, n_points_per_dim)
x3 = torch.linspace(low, high, n_points_per_dim)
x4 = torch.linspace(low, high, n_points_per_dim)
X1, X2, X3, X4 = torch.meshgrid(x1, x2, x3, x4, indexing="ij")
X_grid = torch.stack([X1, X2, X3, X4], dim=-1)
y_grid = FUNCTIONS[function_name](X_grid)

# Nur die Dimensionen 0 (x1) und 1 (x2) betrachten
d1, d2 = 0, 1
mid_idx = n_points_per_dim // 2

fig, ax = plt.subplots(figsize=(8, 6))

# Slice-Logik: x1 und x2 voll, x3 und x4 fixiert auf mid_idx
idx = [mid_idx] * 4
idx[d1] = slice(None)
idx[d2] = slice(None)

# Daten extrahieren
X = X_grid[tuple(idx + [d1])].cpu()
Y = X_grid[tuple(idx + [d2])].cpu()
Z = y_grid[tuple(idx)].cpu()

# Plot Landscape
cp = ax.contourf(X, Y, Z, levels=50, cmap='viridis')
fig.colorbar(cp, ax=ax, label='Funktionswert')

# Plot Scatter (Testpunkte)
x_test_fixed = split.raw_x_test
ax.scatter(
    x_test_fixed[:, d1].cpu(), 
    x_test_fixed[:, d2].cpu(), 
    c=pred.cpu(), 
    s=20, 
    edgecolors='white', 
    linewidth=0.5
)

ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title(f"Schnitt: x1 vs x2 (x3, x4 fixiert auf Index {mid_idx})")

plt.show()